# 📖 Notebook 4: Consistent Hashing

In Notebook 3 we built sticky sessions with `hash(session_id) % N`. That works
fine **until N changes**. The moment you add or remove a backend, almost every
user is reshuffled to a different server — losing every in-memory session,
busting every local cache, breaking every WebSocket.

**Consistent hashing** fixes that. It's the algorithm behind:

- Memcached / Redis client-side sharding
- DynamoDB, Cassandra partition placement
- CDN cache routing (e.g., Akamai)
- Envoy's `RING_HASH` and `MAGLEV` load-balancing policies

## Learning Objectives

By the end of this notebook you'll understand:

- Why naive `hash % N` is a disaster when the fleet size changes
- How a hash **ring** with **virtual nodes** keeps remapping small
- How to implement a tiny consistent-hash router in ~30 lines


## 🛠️ Setup

From the lab folder:

```bash
cd 01-foundations/load-balancing
uv sync
```

### Kernel selection

In VS Code, click the kernel picker at the **top-right** of this notebook and
choose the `.venv` interpreter (it will be named something like
`.venv (Python 3.x)`).

If the `.venv` kernel doesn't appear in the list, reload the VS Code window:

- `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
- Type and select **"Developer: Reload Window"**

No Docker or external services are needed for this lab — everything runs
in-process with plain Python.


## 😬 The naive way: `hash(key) % N`

Suppose you have 4 cache servers and 10,000 keys. You do:

```python
server = servers[hash(key) % 4]
```

Now you add a 5th server. The divisor is now 5 instead of 4, so `hash(key) % 4`
and `hash(key) % 5` give different answers for **most** keys. About **80%** of
keys get remapped to a new server, all caches go cold at once, and your
database melts.

Let's measure that.


In [ ]:
import hashlib

def stable_hash(key: str) -> int:
    """Deterministic hash — Python's built-in hash() is process-randomized."""
    return int(hashlib.md5(key.encode()).hexdigest(), 16)


def naive_assign(keys, servers):
    """Map each key to a server via hash % N."""
    n = len(servers)
    return {k: servers[stable_hash(k) % n] for k in keys}


keys = [f"user-{i}" for i in range(10_000)]
old_servers = [f"cache-{i}" for i in range(4)]
new_servers = old_servers + ["cache-4"]   # add one server

before = naive_assign(keys, old_servers)
after  = naive_assign(keys, new_servers)

moved = sum(1 for k in keys if before[k] != after[k])
print(f"Naive hash%N: {moved}/{len(keys)} keys ({moved/len(keys):.0%}) moved when we added 1 server 😱")

# Going from N to N+1 servers, a key keeps its home only if hash%N == hash%(N+1),
# which happens for roughly 1 key in N+1 -> about (N)/(N+1) = 80% of keys move.
assert 0.75 < moved / len(keys) < 0.85, moved / len(keys)

## 🟢 The fix: a hash ring

Imagine a clock face numbered 0..2³²−1. We:

1. Hash each **server** to a position on the ring (e.g., `hash("cache-2")`).
2. Hash each **key** to a position on the ring.
3. To find a key's server, walk **clockwise** from the key's position until
   you hit a server.

Now when you add a new server, only the **arc** of keys between the new server
and the next server clockwise gets remapped. Everyone else stays put.

To keep the load even (otherwise some servers might own a tiny arc and others a
huge one), each physical server is placed at **many** positions on the ring —
called **virtual nodes** (vnodes). 100–500 vnodes per server is typical.


In [ ]:
import bisect

class ConsistentHashRing:
    """A minimal consistent-hash ring with virtual nodes."""

    def __init__(self, servers: list[str], vnodes: int = 150):
        self.vnodes = vnodes
        # Sorted list of (position, server) and a parallel list of positions
        # so we can use bisect for O(log n) lookups.
        self._ring: list[tuple[int, str]] = []
        for s in servers:
            self._add(s)
        self._sort()

    def _add(self, server: str) -> None:
        for v in range(self.vnodes):
            pos = stable_hash(f"{server}#{v}")
            self._ring.append((pos, server))

    def _sort(self) -> None:
        self._ring.sort()
        self._positions = [p for p, _ in self._ring]

    def add_server(self, server: str) -> None:
        self._add(server)
        self._sort()

    def remove_server(self, server: str) -> None:
        self._ring = [(p, s) for p, s in self._ring if s != server]
        self._sort()

    def get(self, key: str) -> str:
        """Find the server responsible for `key`."""
        h = stable_hash(key)
        # Walk clockwise: smallest position >= h, wrapping around.
        idx = bisect.bisect_right(self._positions, h) % len(self._ring)
        return self._ring[idx][1]


# Sanity check
ring = ConsistentHashRing(["cache-0", "cache-1", "cache-2", "cache-3"])
for k in ["user-1", "user-42", "user-99"]:
    print(f"{k} -> {ring.get(k)}")

# The ring must be deterministic and must never hand back an unknown server.
servers = {"cache-0", "cache-1", "cache-2", "cache-3"}
assert all(ring.get(k) in servers for k in keys)
assert all(ring.get(k) == ring.get(k) for k in keys[:100])
assert len(ring._ring) == 4 * ring.vnodes

## 📉 Measuring the remap rate

Now let's repeat the experiment: 10,000 keys across 4 servers, then add a 5th.
With consistent hashing, only a small fraction of keys should move.


In [ ]:
ring = ConsistentHashRing(["cache-0", "cache-1", "cache-2", "cache-3"])
before = {k: ring.get(k) for k in keys}

ring.add_server("cache-4")
after = {k: ring.get(k) for k in keys}

moved = sum(1 for k in keys if before[k] != after[k])
print(f"Consistent hash: {moved}/{len(keys)} keys ({moved/len(keys):.1%}) moved when we added 1 server 🎉")
print(f"Theoretical ideal: ~{1/5:.0%} (one server's share of the ring)")

# Only the new server's share of the ring may move — nothing bounces between two
# *existing* servers. That is the property the ring actually guarantees.
assert 0.15 < moved / len(keys) < 0.26, moved / len(keys)
assert all(after[k] == "cache-4" for k in keys if before[k] != after[k]), \
    "keys moved between two old servers — that should never happen"

### …and removing one

Adding a server is the easy direction. The one that actually happens at 3 AM is a
server **leaving** — a crash, a scale-down, a rolling deploy. The guarantee has to
hold there too: only the departing server's keys may move, and they must spread
across the survivors rather than all landing on one neighbour.

In [ ]:
ring = ConsistentHashRing([f"cache-{i}" for i in range(5)])
before = {k: ring.get(k) for k in keys}

ring.remove_server("cache-2")
after = {k: ring.get(k) for k in keys}

moved = sum(1 for k in keys if before[k] != after[k])
print(f"Removing 1 of 5 servers moved {moved}/{len(keys)} keys ({moved/len(keys):.1%});"
      f" ideal is ~{1/5:.0%}")

# 1. Only ~1/N of keys move at all.
assert 0.15 < moved / len(keys) < 0.26, moved / len(keys)
# 2. Every key that moved was homed on the departed server. Nobody else is disturbed.
assert all(before[k] == "cache-2" for k in keys if before[k] != after[k])
# 3. The orphans are spread over the survivors, not dumped on one unlucky neighbour.
from collections import Counter
inherited = Counter(after[k] for k in keys if before[k] == "cache-2")
print("who inherited cache-2's keys:", dict(sorted(inherited.items())))
assert len(inherited) == 4, "orphaned keys should land on all four survivors"
assert max(inherited.values()) < 2 * min(inherited.values())

# Compare with the naive scheme, where removing a server reshuffles almost everything.
naive_before = naive_assign(keys, [f"cache-{i}" for i in range(5)])
naive_after = naive_assign(keys, [f"cache-{i}" for i in range(4)])
naive_moved = sum(1 for k in keys if naive_before[k] != naive_after[k])
print(f"\nSame removal with hash%N: {naive_moved/len(keys):.0%} of keys move.")
assert naive_moved / len(keys) > 0.7

## 📊 Side-by-side

Let's chart what happens as we add servers, one at a time, with both
strategies.


In [ ]:
import matplotlib.pyplot as plt

def remap_rate_naive(start_n: int, add: int) -> list[float]:
    rates = []
    prev_servers = [f"cache-{i}" for i in range(start_n)]
    prev = naive_assign(keys, prev_servers)
    for i in range(add):
        prev_servers.append(f"cache-{start_n + i}")
        new = naive_assign(keys, prev_servers)
        rates.append(sum(1 for k in keys if prev[k] != new[k]) / len(keys))
        prev = new
    return rates


def remap_rate_ring(start_n: int, add: int) -> list[float]:
    rates = []
    ring = ConsistentHashRing([f"cache-{i}" for i in range(start_n)])
    prev = {k: ring.get(k) for k in keys}
    for i in range(add):
        ring.add_server(f"cache-{start_n + i}")
        new = {k: ring.get(k) for k in keys}
        rates.append(sum(1 for k in keys if prev[k] != new[k]) / len(keys))
        prev = new
    return rates


naive_rates = remap_rate_naive(4, 6)
ring_rates  = remap_rate_ring(4, 6)
labels = [f"+server #{5+i}" for i in range(6)]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(labels))
ax.bar([i - 0.2 for i in x], naive_rates, width=0.4, label="naive hash % N", color="#FF5630")
ax.bar([i + 0.2 for i in x], ring_rates,  width=0.4, label="consistent hashing", color="#36B37E")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("fraction of keys remapped")
ax.set_title("Cost of adding one server, naive vs consistent")
ax.legend()
plt.tight_layout()
plt.show()


## 🎚️ Why virtual nodes matter

With `vnodes=1` each server sits at exactly one spot on the ring, and random points
on a ring do **not** divide it into equal arcs — one server can easily own three
times as many keys as another. Adding more vnodes averages those arcs out.

The improvement is dramatic up to a few dozen vnodes, then flattens: once the arcs
are even, what's left is just sampling noise from having only 10,000 keys across 4
servers. That plateau is why "150 vnodes per server" is a folk constant rather than
a number anyone tunes.

In [ ]:
from collections import Counter

def load_distribution(vnodes: int) -> dict[str, int]:
    ring = ConsistentHashRing([f"cache-{i}" for i in range(4)], vnodes=vnodes)
    counts = Counter(ring.get(k) for k in keys)
    return dict(sorted(counts.items()))


imbalance = {}
for v in (1, 10, 50, 150, 500):
    dist = load_distribution(v)
    lo, hi = min(dist.values()), max(dist.values())
    imbalance[v] = hi / lo
    print(f"vnodes={v:4d}  per-server: {dist}  imbalance hi/lo = {hi/lo:.2f}x")

# One vnode per server is badly skewed; a realistic setting is within ~30% of even.
assert imbalance[1] > 2.0, imbalance[1]
assert imbalance[150] < 1.3, imbalance[150]
# The big win is from 1 -> 50; past that it is noise, not a trend.
assert imbalance[50] < 0.6 * imbalance[1]
assert imbalance[500] < 0.6 * imbalance[10]

## 🧠 Takeaways

- **`hash(key) % N`** moves ~`(N-1)/N` of all keys whenever `N` changes —
  unacceptable for caches, sharded databases, or sticky sessions.
- **Consistent hashing** moves only the keys that "belong" to the changed
  server (~`1/N` of keys). The rest stay put.
- **Virtual nodes** (~150/server) make the load distribution even.
- Use **deterministic hashes** (`hashlib.md5`, `xxhash`, `murmur3`), not
  Python's randomized `hash()`.

### Where you'll meet this in real life

- **Memcached / Redis client libs** (`libketama`, `redis-py-cluster`)
- **Cassandra, ScyllaDB, DynamoDB** — partition keys are placed on a ring
- **Envoy proxy** — `RING_HASH` policy for sticky routing
- **CDNs** — cache keys (URLs) routed to a specific edge POP

### What's next

That wraps up the load-balancing foundations. From here, dive into:

- **NGINX `upstream` blocks** — try `least_conn` and `hash $remote_addr`
- **Envoy** — read the `cluster` config, especially `LEAST_REQUEST` and
  `RING_HASH`
- **AWS ALB / NLB** — see which of these algorithms each one uses
- The lab on **caching** (later in `01-foundations/`) which builds on the same
  consistent-hashing idea for distributing cache keys
